# 00 — Data Preparation
### Shared preprocessing pipeline used by all models
Run this notebook FIRST before any model notebook.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('✅ Imports complete.')

In [ ]:
# ── FILE PATH ────────────────────────────────────────────────────────
DATA_FILE = r'C:\Users\DeelakaD\OneDrive - MAS Holdings (Pvt) Ltd\Documents\GitHub\Preventing-Mechanism\Datasets\Combined_Machine_Data.xlsx'  # ← UPDATE THIS PATH

# ── SETTINGS ─────────────────────────────────────────────────────────
TIME_STEPS = 7

KNOWN_BREAKDOWNS = [
    'Needle Breakages',
    'High Foot Pressure',
    'Cut/Needle Hole',
    'Thread Breakages',
    'Pneumatic Issues',
    'Thread Jamming',
    'Code Uneven',
    'Roping',
    'Oil Mark',
    'Skip Stitches/Slip',
    'Gathering/Puckering',
    'Waveness',
    'Binding/Seam Open',
    'Blade Blunt',
]
ALLOWED_STATES = ['Healthy'] + KNOWN_BREAKDOWNS

# ── FREQUENCY BAND GROUPS (for FAG-TFT) ─────────────────────────────
# Physically motivated grouping of vibration frequency bands
VIB_GROUP_1 = [f'{i}-{i+10}Hz' for i in range(10,  100, 10)]   # 10-100Hz
VIB_GROUP_2 = [f'{i}-{i+10}Hz' for i in range(100, 300, 10)]   # 100-300Hz
VIB_GROUP_3 = [f'{i}-{i+10}Hz' for i in range(300, 610, 10)]   # 300-610Hz
ELEC_FEATS  = ['machineVoltageMean','machineVoltageMin','machineVoltageMax',
               'machineCurrentMean','machineCurrentMin','machineCurrentMax']

print(f'✅ Config loaded.')
print(f'   Known breakdowns    : {len(KNOWN_BREAKDOWNS)}')
print(f'   VIB Group 1 (10-100Hz)  : {len(VIB_GROUP_1)} bands')
print(f'   VIB Group 2 (100-300Hz) : {len(VIB_GROUP_2)} bands')
print(f'   VIB Group 3 (300-610Hz) : {len(VIB_GROUP_3)} bands')
print(f'   Electrical features     : {len(ELEC_FEATS)}')

In [ ]:
# ── LOAD DATA ────────────────────────────────────────────────────────
master_df = pd.read_excel(DATA_FILE)

# Filter valid vibration records (must start with '10')
master_df = master_df[master_df['machineVibration'].astype(str).str.startswith('10')].copy()

# Fill empty Breakdown column as 'Healthy'
master_df['Breakdown'] = master_df['Breakdown'].fillna('Healthy')

# Keep only allowed states
master_df = master_df[master_df['Breakdown'].isin(ALLOWED_STATES)].reset_index(drop=True)

print(f'✅ Data loaded: {len(master_df)} total records')
print(f'\nClass distribution:')
print(master_df['Breakdown'].value_counts())

In [12]:
# ── FEATURE EXTRACTION ───────────────────────────────────────────────
def extract_features(df):
    vib_records = []
    for val in df['machineVibration']:
        vib_dict = {}
        parts = str(val).split(',')
        try:
            for i in range(0, len(parts)-1, 2):
                f_start = int(parts[i])
                vib_dict[f'{f_start}-{f_start+10}Hz'] = int(parts[i+1])
        except: pass
        vib_records.append(vib_dict)

    expected_vib = [f'{i}-{i+10}Hz' for i in range(10, 610, 10)]
    vib_df  = pd.DataFrame(vib_records).reindex(columns=expected_vib, fill_value=0)
    elec_df = df[ELEC_FEATS].ffill().fillna(0).reset_index(drop=True)
    return pd.concat([vib_df.reset_index(drop=True), elec_df], axis=1)

X_all = extract_features(master_df)
y_all = master_df['Breakdown'].values
print(f'✅ Features extracted: {X_all.shape}  (samples x 66 features)')

✅ Features extracted: (5374, 66)  (samples x 66 features)


In [13]:
# ── ENCODE LABELS ────────────────────────────────────────────────────
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y_all)
num_classes = len(encoder.classes_)
print(f'✅ Classes: {list(encoder.classes_)}')

# ── TRAIN/TEST SPLIT ─────────────────────────────────────────────────
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_all.values, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

# ── SCALE (fit on train only) ────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled  = scaler.transform(X_test_raw)

print(f'   Train: {X_train_scaled.shape}  Test: {X_test_scaled.shape}')

✅ Classes: ['Blade Blunt', 'Healthy', 'High Foot Pressure', 'Skip Stitches/Slip', 'Waveness']
   Train: (4299, 66)  Test: (1075, 66)


In [14]:
# ── SEQUENCE CREATION ────────────────────────────────────────────────
def create_sequences(X, y, time_steps):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:i+time_steps])
        ys.append(y[i+time_steps])
    return np.array(Xs), np.array(ys)

X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train, TIME_STEPS)
X_test_seq,  y_test_seq  = create_sequences(X_test_scaled,  y_test,  TIME_STEPS)

print(f'✅ Sequences created.')
print(f'   Train: {X_train_seq.shape}  →  (samples, {TIME_STEPS}, 66)')
print(f'   Test : {X_test_seq.shape}')

✅ Sequences created.
   Train: (4292, 7, 66)  →  (samples, 7, 66)
   Test : (1068, 7, 66)


In [15]:
# ── HEALTHY-ONLY DATA FOR AUTOENCODER ────────────────────────────────
healthy_mask = master_df['Breakdown'].values == 'Healthy'
X_healthy    = extract_features(master_df[healthy_mask])
X_healthy_scaled = scaler.transform(X_healthy.values)

def create_sequences_X(X, time_steps):
    return np.array([X[i:i+time_steps] for i in range(len(X)-time_steps)])

X_healthy_seq = create_sequences_X(X_healthy_scaled, TIME_STEPS)
split = int(len(X_healthy_seq)*0.8)
X_ae_train, X_ae_test = X_healthy_seq[:split], X_healthy_seq[split:]

print(f'✅ Autoencoder sequences: train={X_ae_train.shape}  test={X_ae_test.shape}')

✅ Autoencoder sequences: train=(3952, 7, 66)  test=(988, 7, 66)


In [16]:
# ── SAVE ALL PREPARED DATA ───────────────────────────────────────────
prepared = {
    'X_train_seq'  : X_train_seq,
    'X_test_seq'   : X_test_seq,
    'y_train_seq'  : y_train_seq,
    'y_test_seq'   : y_test_seq,
    'X_ae_train'   : X_ae_train,
    'X_ae_test'    : X_ae_test,
    'num_classes'  : num_classes,
    'num_features' : X_train_seq.shape[2],
    'TIME_STEPS'   : TIME_STEPS,
    'VIB_GROUP_1'  : VIB_GROUP_1,
    'VIB_GROUP_2'  : VIB_GROUP_2,
    'VIB_GROUP_3'  : VIB_GROUP_3,
    'ELEC_FEATS'   : ELEC_FEATS,
    'feature_names': list(X_all.columns),
}
with open('prepared_data.pkl','wb') as f: pickle.dump(prepared, f)
with open('scaler.pkl','wb') as f: pickle.dump(scaler, f)
with open('encoder.pkl','wb') as f: pickle.dump(encoder, f)

print('✅ Saved: prepared_data.pkl  scaler.pkl  encoder.pkl')
print(f'   num_classes  : {num_classes}')
print(f'   num_features : {X_train_seq.shape[2]}')
print(f'   TIME_STEPS   : {TIME_STEPS}')

✅ Saved: prepared_data.pkl  scaler.pkl  encoder.pkl
   num_classes  : 5
   num_features : 66
   TIME_STEPS   : 7
